In [20]:
import requests
from collections import defaultdict

In [ ]:
OFFICIAL_URL = "https://www.panynj.gov/bin/portauthority/ridepath.json"

STATION_CODE = "JSQ"
DESTINATION = "ToNY"
ROUTE = {"33rd Street", "World Trade Center"}

r = requests.get(OFFICIAL_URL)
data = r.json()


route_colors = {}
for route in ROUTE:
    route_colors[route] = next(
        (
            m["lineColor"]
            for s in data["results"]
            for d in s.get("destinations", [])
            for m in d.get("messages", [])
            if m["headSign"] == route
        ),
        None,
    )




station = next(s for s in data["results"] if s["consideredStation"] == STATION_CODE)
destination = next(d for d in station["destinations"] if d["label"] == DESTINATION)



#group trains by route
route_trains = defaultdict(list)
for train in destination["messages"]:
    route_trains[train['headSign']].append(int(train["secondsToArrival"]) // 60)

for route in route_trains:
    route_trains[route].sort()



In [50]:
route_trains

defaultdict(list, {'World Trade Center': [1, 7], '33rd Street': [3, 8]})

In [ ]:
import IPython.display as display
from IPython.display import HTML

def render_path_signs(route_trains):
    # Colors for each route (fallback: gray)
    PATH_COLORS = {
        "World Trade Center": "#0f4ba8",  # Blue
        "33rd Street": "#008c39",         # Green
    }
    sign_html = ""
    for route, arrivals in route_trains.items():
        color = PATH_COLORS.get(route, "#555")
        arrivals_str = " ".join(f"<span style='background: #fff; color: #000; border-radius:2px; font-variant-numeric: tabular-nums; padding: 0 4px; margin-left: 3px; font-size: 16px; border: 1px solid #ccc'>{m}'</span>" for m in arrivals)
        sign_html += f"""
        <div style='display:flex;align-items:center;margin-bottom:12px'>
            <span style='background:{color};color:white;border-radius:7px;padding: 6px 16px; font-weight: bold; font-size: 18px;letter-spacing:1.5px; min-width:120px; display: inline-block; text-align: center'>{route.upper()}</span>
            <span style='margin-left:18px; font-size:16px;'>{"min" if arrivals else ""} {arrivals_str}</span>
        </div>
        """
    display.display(HTML(f"<div>{sign_html}</div>"))

render_path_signs(route_trains)

ModuleNotFoundError: No module named 'PIL'